In [35]:
!pip install -q "datasets<3.0.0" soundfile accelerate torch resemblyzer librosa pandas scipy edge-tts nest_asyncio

In [36]:
import asyncio
import edge_tts
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample
from resemblyzer import VoiceEncoder, preprocess_wav

# Configuration
NUM_SAMPLES = 30
EDGE_VOICE = "ur-PK-AsadNeural"

# Initialize embedding model
encoder = VoiceEncoder()

async def tts_to_file(text, out_path, voice=EDGE_VOICE):
    communicate = edge_tts.Communicate(text=text, voice=voice)
    await communicate.save(out_path)

def synthesize_edgetts(text, out_path, voice=EDGE_VOICE):
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        loop.run_until_complete(tts_to_file(text, out_path, voice))
    else:
        loop.run_until_complete(tts_to_file(text, out_path, voice))

def run_evaluation(dataset_name, samples, text_key, audio_key="audio"):
    base_dir = Path(f"eval_{dataset_name}")
    ref_dir = base_dir / "reference"
    gen_dir = base_dir / "generated_edgetts"
    anc_dir = base_dir / "anchor"

    for d in [ref_dir, gen_dir, anc_dir]:
        d.mkdir(parents=True, exist_ok=True)

    results = []

    if hasattr(samples, "select"):
        iterable_samples = samples.select(range(min(NUM_SAMPLES, len(samples))))
    else:
        iterable_samples = samples[:NUM_SAMPLES]

    for i, sample in enumerate(iterable_samples):
        text = sample[text_key]

        # Reference audio
        audio = sample[audio_key]
        ref_speech = audio["array"]
        ref_sr = audio["sampling_rate"]

        if len(ref_speech.shape) > 1:
            ref_speech = np.mean(ref_speech, axis=1)

        sample_id = f"{dataset_name}_{i+1:02d}"
        ref_path = ref_dir / f"{sample_id}.wav"
        gen_path = gen_dir / f"{sample_id}.wav"
        anc_path = anc_dir / f"{sample_id}.wav"

        sf.write(ref_path, ref_speech, ref_sr)

        # EdgeTTS generation
        synthesize_edgetts(text, str(gen_path), EDGE_VOICE)
        gen_speech, gen_sr = sf.read(gen_path)

        if len(gen_speech.shape) > 1:
            gen_speech = np.mean(gen_speech, axis=1)

        # Anchor creation: degrade ref by downsampling to 4k and upsampling back
        downsampled = resample(ref_speech, int(len(ref_speech) * 4000 / ref_sr))
        upsampled = resample(downsampled, len(ref_speech))
        sf.write(anc_path, upsampled, ref_sr)

        # Embeddings + similarities
        ref_wav = preprocess_wav(ref_speech, ref_sr)
        gen_wav = preprocess_wav(gen_speech, gen_sr)

        ref_emb = encoder.embed_utterance(ref_wav)
        gen_emb = encoder.embed_utterance(gen_wav)

        sim = np.dot(ref_emb, gen_emb) / (np.linalg.norm(ref_emb) * np.linalg.norm(gen_emb))
        self_sim = np.dot(ref_emb, ref_emb) / (np.linalg.norm(ref_emb) * np.linalg.norm(ref_emb))

        results.append({
            "dataset": dataset_name,
            "sample_id": sample_id,
            "urdu_text": text,
            "audio_duration_sec": len(ref_speech) / ref_sr,
            "ref_gen_similarity": float(sim),
            "ref_self_similarity": float(self_sim)
        })

    return pd.DataFrame(results)

Loaded the voice encoder model on cuda in 0.02 seconds.


In [37]:
from datasets import load_dataset
ds_fleurs_stream = load_dataset(
    "google/fleurs", "ur_pk", split="train", trust_remote_code=True, streaming=True
)

# Apply filter: gender == 0 is male
ds_fleurs_male = ds_fleurs_stream.filter(lambda x: x["gender"] == 0)

fleurs_samples = []
for sample in ds_fleurs_male:
    fleurs_samples.append(sample)
    if len(fleurs_samples) >= NUM_SAMPLES:
        break
results_fleurs = run_evaluation("fleurs", fleurs_samples, "transcription")

In [38]:
from google.colab import userdata
try:
    hf_token = userdata.get('hfToken')
except:
    hf_token = None

In [ ]:
# Target Urdu directory
from datasets import load_dataset
ds_urduspeech_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True
)

# Take first NUM_SAMPLES with style = "WIKI" and gender = "Male"
urduspeech_samples = []
for sample in ds_urduspeech_stream:
    if sample.get("style") == "WIKI" and sample.get("gender") == "Male":   # filter by style and gender
        urduspeech_samples.append(sample)
    if len(urduspeech_samples) >= NUM_SAMPLES:
        break

results_urduspeech = run_evaluation("urduspeech", urduspeech_samples, "text")

In [40]:
# all_data = []
all_data = pd.concat([
    results_fleurs,
    results_urduspeech
], ignore_index=True)

df = pd.DataFrame(all_data)
df = df[['dataset', 'sample_id', 'urdu_text', 'audio_duration_sec', 'ref_gen_similarity', 'ref_self_similarity']]

# Summary Table
stats = df.groupby('dataset').agg({
    'ref_gen_similarity': 'mean',
    'ref_self_similarity': 'mean'
})

stats.columns = ['mean_gen_sim', 'mean_self_sim']
stats['embedding_gap'] = stats['mean_self_sim'] - stats['mean_gen_sim']

print("=== GLOBAL SUMMARY STATISTICS ===")
display(stats)

df.to_csv("urdu_edgetts_evaluation.csv", index=False, encoding='utf-8-sig')
print("\nFull results saved to urdu_tts_evaluation_all_datasets.csv")

=== GLOBAL SUMMARY STATISTICS ===


,mean_gen_sim,mean_self_sim,embedding_gap
dataset,,,
fleurs,0.530325,1.0,0.469675
urduspeech,0.770126,1.0,0.229874



Full results saved to urdu_tts_evaluation_all_datasets.csv


In [41]:
import zipfile
datasets = ["fleurs", "urduspeech"]

with zipfile.ZipFile("mushra_FORMAL_datasets.zip", "w") as zf:
    for d_name in datasets:
        folder = f"eval_{d_name}"
        for root, dirs, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))

print("Successfully created mushra_all_datasets.zip containing all audio samples.")

Successfully created mushra_all_datasets.zip containing all audio samples.
